# Mallory, Hardcastle et al. (2021): the LN counts under reading (1b) random folds, (2a) drop

Companion to `reproduce_mallory_2021.ipynb` (dandiset 000053, version 0.210819.0345; the paper,
the claims file and the choice of version are described there). The claims file leaves two details
of the LN pipeline open and asks for both readings of each: (1) how the 10-s sections are assigned
to the 10 folds, (a) in turn as compute_test_train_ind.m does, or (b) at random; (2) whether bins
outside a variable's inner 98% are (a) dropped or (b) clipped to the range. The main notebook runs
(1a, 2a). This one runs (1b, 2a), the forward search only (the reading (1a, 2b) was planned in a second
notebook and not run, for lack of time), for the counts of
fig4j-eye-encoding-cells, fig4k-eye-conjunctive and fig7c-head-fixed-model-fit, on the same 11
recordings (selected by the link claim of the main notebook, repeated here).

In [ ]:
import os
import time
import resource
import warnings

import numpy as np
import pandas as pd

import mec_ln as L

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
STARTED = time.time()
SUBSET = int(os.environ.get("MALLORY_SUBSET", "0"))   # 0: every unit (develop on a few)
urls = L.asset_urls("ecephys")
SESSIONS, fsd = L.paper_sessions(urls)
print("Sessions matched to the figshare deposit:", sum(p is not None for p in SESSIONS), "of", len(SESSIONS))

In [ ]:
DATA = {p: L.load_session(urls[p]) for p in SESSIONS}
print(pd.DataFrame([{"session": p.split("/")[1][:40], "mouse": s["subject"], "units": s["counts"].shape[0],
                     "frames": len(s["t"]), "repaired": len(s["repaired"]), "spikes_outside": int(s["n_outside"].sum())}
                    for p, s in DATA.items()]).to_string(index=False))
print("Units:", sum(s["counts"].shape[0] for s in DATA.values()))

In [ ]:
READINGS = {k: v for k, v in {"(1b) random folds, (2a) drop": (L.FOLDS_RANDOM, L.RANGE_DROP),
                               "(1a) folds in turn, (2b) clip": (L.FOLDS_TURN, L.RANGE_CLIP)}.items()
            if k == "(1b) random folds, (2a) drop"}
OUT = {}
for name, (fmode, rmode) in READINGS.items():
    t0 = time.time()
    rows = []
    for si, p in enumerate(SESSIONS):
        s = DATA[p]
        prep = L.prepare(s, rmode)
        folds = L.fold_labels(prep["n_keep"], fmode, seed=2000 + si)
        P = L.by_fold(prep, folds)
        Y = s["counts"][:, prep["keep"]]
        if SUBSET:
            Y = Y[:SUBSET]
        res = L.analyse_session(P, Y, processes=2, with_extras=False)
        for r in res:
            rows.append({"mouse": s["subject"], "session": p, "encoded": r["encoded"],
                         "perf": float(np.nanmean(r["sel_fits"])) if r["encoded"] else np.nan,
                         **{v: v in r["encoded"] for v in L.VARS}})
        print(f"  {name}: {p.split('/')[1][:30]} {prep['n_keep']} of {prep['n_valid']} bins")
    OUT[name] = pd.DataFrame(rows)
    print(f"{name}: {len(rows)} units in {time.time() - t0:.0f} s")
print(f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

### fig4j-eye-encoding-cells, other readings

In [ ]:
EXPECTED_4J = {"eye_any": (711, 71.1), "Eh": (343, 34.3), "Ev": (234, 23.4), "Ehd": (244, 24.4), "Evd": (135, 13.5)}
for name, c in OUT.items():
    c["eye_any"] = c[L.EYE].any(axis=1)
    tab = pd.DataFrame([{"count": k, "measured": int(c[k].sum()), "paper": v, "tol": t,
                         "within": abs(int(c[k].sum()) - v) <= t} for k, (v, t) in EXPECTED_4J.items()])
    print(name, f"({len(c)} units); per-variable counts within tolerance: {int(tab.within[1:].sum())} of 4")
    print(tab.to_string(index=False))
    print("  by mouse, fraction with any eye variable:", c.groupby("mouse").eye_any.mean().round(3).to_dict())

**Result: under this reading, outside tolerance.** 614 units encode an eye variable (paper 711, tolerance 71.1); only the Eh dot count, 231, is within tolerance. The claim is judged in the main notebook, where the reading with folds in turn matches.

### fig4k-eye-conjunctive, other readings

In [ ]:
for name, c in OUT.items():
    e = c[c.eye_any]
    n_eye = e[L.EYE].sum(axis=1)
    b_alone = c.encoded.apply(lambda x: x == ["B"]).mean()
    print(f"{name}: eye cells also B or Bs {(e.B | e.Bs).mean():.3f} (paper 0.96, tol 0.096); more than one eye "
          f"variable {(n_eye > 1).mean():.3f} (paper 0.28, tol 0.028); B alone {b_alone:.3f} of all (paper 0.23, tol 0.023)")

**Result: under this reading, two of three within tolerance.** 0.964 of eye cells also encode B or Bs and 0.232 of all units encode B alone; 0.238 encode more than one eye variable (paper 0.28, tolerance 0.028).

### fig7c-head-fixed-model-fit, other readings

In [ ]:
for name, c in OUT.items():
    perf = c.perf.dropna()
    print(f"{name}: {len(perf)} cells encode a variable (paper 2191, tol 219.1); median llh increase "
          f"(compute_llh.m) {perf.median():.4f} [{perf.quantile(0.25):.4f} to {perf.quantile(0.75):.4f}] "
          f"(paper 0.05, tol 0.005); in bits per spike {perf.median() / L.LOG2 ** 2:.4f}")

**Result: under this reading, the median is within tolerance and the count just outside.** 1964 units encode a variable (paper 2191; the lower limit is 1971.9, so the count misses by 8 units) and the median is 0.0534 (paper 0.05, tolerance 0.005).

In [ ]:
print(f"Notebook time: {(time.time() - STARTED) / 60:.1f} min")